# Stage 1 — Shared micro spatial E1

Train one MobileNetV3-Small spatial baseline on the frozen micro Train set. ImageNet backbone weights are mandatory; the E1 task head starts randomly. The threshold is calibrated on micro-Val and then locked for LCC external-dev evaluation.


In [ ]:
# Edit only the paths that cannot be resolved uniquely under /kaggle/input.
STAGE0_RESOURCE_DIR = ""
CELEBA_ROOT = ""
LCC_FASD_ROOT = ""
IMAGENET_WEIGHTS_PATH = ""  # Optional torchvision MobileNetV3-Small state_dict
OUTPUT_ROOT = "/kaggle/working/pad_micro_search_v1"
SEED = 42
BATCH_SIZE = 128
NUM_WORKERS = 4
AMP = True


In [ ]:
import csv
import hashlib
import json
import math
import os
import random
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import MobileNet_V3_Small_Weights

CELEBA_MEAN = np.asarray([0.5931, 0.4690, 0.4229], dtype=np.float32)
CELEBA_STD = np.asarray([0.2471, 0.2214, 0.2157], dtype=np.float32)
INPUT_SIZE = 224
E1_ARCHITECTURE = "MobileNetV3-Small-GAP-576-256-BN-HSwish-Dropout02-128-3-v1"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def find_unique_file(name, root="/kaggle/input", explicit_dir=""):
    base = Path(explicit_dir).expanduser() if explicit_dir else Path(root)
    if explicit_dir and base.is_file():
        matches = [base.resolve()] if base.name == name else []
    else:
        matches = sorted(p.resolve() for p in base.rglob(name) if p.is_file()) if base.exists() else []
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {name}, found {len(matches)}. Set an explicit directory. "
            f"Examples: {[str(p) for p in matches[:8]]}"
        )
    return matches[0]

def find_stage0_dir():
    if STAGE0_RESOURCE_DIR:
        root = Path(STAGE0_RESOURCE_DIR).expanduser().resolve()
        if not (root / "stage0_summary.json").is_file():
            if (root / "stage0" / "stage0_summary.json").is_file():
                root = root / "stage0"
            else:
                raise FileNotFoundError(f"stage0_summary.json not under {root}")
        return root
    matches = sorted(p.parent.resolve() for p in Path("/kaggle/input").rglob("stage0_summary.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one Stage 0 resource directory, found {len(matches)}; set STAGE0_RESOURCE_DIR")
    return matches[0]

def find_celeba_root():
    if CELEBA_ROOT:
        root = Path(CELEBA_ROOT).expanduser().resolve()
        if not (root / "metas/intra_test/train_label.json").is_file():
            raise FileNotFoundError(root / "metas/intra_test/train_label.json")
        return root
    matches = []
    for p in Path("/kaggle/input").rglob("train_label.json"):
        if p.as_posix().endswith("metas/intra_test/train_label.json"):
            matches.append(p.parent.parent.parent.resolve())
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one CelebA-Spoof root, found {len(matches)}; set CELEBA_ROOT")
    return matches[0]

def find_lcc_root(manifest_path):
    if LCC_FASD_ROOT:
        root = Path(LCC_FASD_ROOT).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root
    with open(manifest_path, newline="", encoding="utf-8") as f:
        first = next(csv.DictReader(f))
    rel_parts = PurePosixPath(first["path"]).parts
    matches = []
    for p in Path("/kaggle/input").rglob(rel_parts[-1]):
        if p.is_file() and tuple(p.parts[-len(rel_parts):]) == rel_parts:
            matches.append(p.resolve().parents[len(rel_parts) - 1])
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(f"Could not uniquely resolve LCC root from {first['path']}; set LCC_FASD_ROOT")
    return matches[0]

CELEBA_BBOX_STATUS_FACTORS = {
    "VALID": 1.55,
    "FALLBACK_DET_FAIL": 1.50,
    "FALLBACK_SUSPICIOUS": 1.50,
}


def validate_bbox_record(record, domain, key):
    if not isinstance(record, dict):
        raise TypeError("BBox record must be a mapping for {}".format(key))
    status = str(record.get("status", ""))
    if domain == "celeba":
        if status == "INVALID":
            raise RuntimeError("Selected micro key has INVALID E1 bbox record: {}".format(key))
        if status not in CELEBA_BBOX_STATUS_FACTORS:
            raise RuntimeError("Unknown/unusable E1 bbox status {} for {}".format(status, key))
        expected_factor = CELEBA_BBOX_STATUS_FACTORS[status]
    elif domain == "lcc":
        if status != "VALID":
            raise RuntimeError("Only Stage 0 LCC VALID records may be scored: {} ({})".format(key, status))
        expected_factor = 1.55
    else:
        raise ValueError("Unknown PAD crop domain: {}".format(domain))
    if "crop_factor" not in record:
        raise KeyError("BBox record is missing crop_factor for {}".format(key))
    try:
        crop_factor = float(record["crop_factor"])
    except (TypeError, ValueError) as exc:
        raise ValueError("Invalid crop_factor for {}: {!r}".format(key, record.get("crop_factor"))) from exc
    if not np.isfinite(crop_factor) or not np.isclose(crop_factor, expected_factor, rtol=0.0, atol=1e-6):
        raise ValueError(
            "BBox status/crop-factor mismatch for {}: status={} factor={} expected={}".format(
                key, status, crop_factor, expected_factor
            )
        )
    bbox = np.asarray(record.get("bbox_xyxy", []), dtype=np.float64)
    if bbox.shape != (4,) or not np.isfinite(bbox).all() or bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
        raise ValueError("Invalid bbox_xyxy for {}: {!r}".format(key, record.get("bbox_xyxy")))
    return crop_factor


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def labels_from_attack_codes(codes):
    codes = np.asarray(codes, dtype=np.int64)
    labels = np.full(codes.shape, 2, dtype=np.int64)
    labels[codes == 0] = 0
    labels[np.isin(codes, np.arange(1, 8))] = 1
    return labels

def load_stage0_resources():
    stage0 = find_stage0_dir()
    manifest_path = stage0 / "micro_manifest_v1_seed42.npz"
    bbox_path = stage0 / "celeba_micro_bbox_cache.json"
    lcc_manifest_path = stage0 / "lcc_external_dev_manifest.csv"
    lcc_cache_path = stage0 / "lcc_scrfd_cache_micro_v1.json"
    fingerprints_path = stage0 / "micro_resource_fingerprints.json"
    summary_path = stage0 / "stage0_summary.json"
    for path in [manifest_path, bbox_path, lcc_manifest_path, lcc_cache_path, fingerprints_path, summary_path]:
        if not path.is_file():
            raise FileNotFoundError("Stage 0 resource missing: {}".format(path))
    fingerprints = json.loads(fingerprints_path.read_text())
    stage0_summary = json.loads(summary_path.read_text())
    if stage0_summary.get("schema") != "pad_micro_stage0_summary_v1":
        raise RuntimeError("Unsupported Stage 0 summary schema")
    for path, key in [
        (manifest_path, "micro_manifest_sha256"),
        (lcc_manifest_path, "lcc_manifest_sha256"),
        (lcc_cache_path, "lcc_cache_sha256"),
        (bbox_path, "celeba_bbox_cache_sha256"),
    ]:
        if sha256_file(path) != fingerprints.get(key):
            raise RuntimeError("Stage 0 SHA256 mismatch for {}".format(key))
    with np.load(manifest_path, allow_pickle=False) as pack:
        train_keys = pack["micro_train_keys"].astype(str)
        val_keys = pack["micro_val_keys"].astype(str)
        train_labels = pack["micro_train_labels"].astype(np.int64)
        val_labels = pack["micro_val_labels"].astype(np.int64)
        train_codes = pack["micro_train_attack_codes"].astype(np.int64)
        val_codes = pack["micro_val_attack_codes"].astype(np.int64)
    if len(train_keys) != 5000 or len(val_keys) != 2000:
        raise RuntimeError("Stage 0 must freeze exactly 5,000 Train and 2,000 Val keys")
    if len(set(train_keys) & set(val_keys)):
        raise RuntimeError("Stage 0 Train/Val keys overlap")
    for name, labels, codes in [
        ("Train", train_labels, train_codes),
        ("Val", val_labels, val_codes),
    ]:
        if labels.shape != codes.shape or set(map(int, labels)) != {0, 1, 2}:
            raise RuntimeError("Stage 0 {} label/code arrays are malformed or miss a class".format(name))
        if not np.array_equal(labels, labels_from_attack_codes(codes)):
            raise RuntimeError("Stage 0 {} labels do not match E1 v5.3 attack-code mapping".format(name))
    bbox_payload = json.loads(bbox_path.read_text())
    bbox_cache = bbox_payload.get("records", {})
    selected_keys = set(map(str, train_keys)) | set(map(str, val_keys))
    if bbox_payload.get("schema") != "celeba_micro_bbox_cache_v1" or set(bbox_cache) != selected_keys:
        raise RuntimeError("Stage 0 CelebA bbox cache keys/schema differ from the frozen micro manifest")
    for key, record in bbox_cache.items():
        validate_bbox_record(record, "celeba", key)
    lcc_df = pd.read_csv(lcc_manifest_path)
    lcc_payload = json.loads(lcc_cache_path.read_text())
    lcc_records = lcc_payload.get("records", {})
    if lcc_payload.get("schema") != "pad_micro_lcc_scrfd_cache_v1":
        raise RuntimeError("Unsupported Stage 0 LCC cache schema")
    if lcc_df["path"].astype(str).duplicated().any():
        raise RuntimeError("Stage 0 LCC manifest contains duplicate paths")
    if not set(lcc_df["label"].astype(int)).issubset({0, 1}):
        raise RuntimeError("Stage 0 LCC labels must be exactly 0=Real or 1=Attack")
    record_by_path = {}
    for record in lcc_records.values():
        path = str(record.get("path", ""))
        record_by_path.setdefault(path, []).append(record)
    for row in lcc_df.itertuples(index=False):
        matches = record_by_path.get(str(row.path), [])
        if len(matches) != 1:
            raise RuntimeError("Stage 0 LCC cache join for {} has {} rows".format(row.path, len(matches)))
        record = matches[0]
        if str(record.get("status", "")) != str(row.status):
            raise RuntimeError("Stage 0 LCC manifest/cache status mismatch for {}".format(row.path))
        if int(record.get("label", -1)) != int(row.label):
            raise RuntimeError("Stage 0 LCC manifest/cache label mismatch for {}".format(row.path))
        if str(record.get("frame_id", "")) != str(row.record_id):
            raise RuntimeError("Stage 0 LCC manifest record_id mismatch for {}".format(row.path))
        if str(row.status) == "VALID":
            validate_bbox_record(record, "lcc", row.path)
    valid_paths = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    valid_labels = lcc_df.loc[lcc_df["status"].astype(str) == "VALID", "label"].astype(int).tolist()
    if not valid_paths or set(valid_labels) != {0, 1}:
        raise RuntimeError("Common LCC VALID set must be non-empty and contain Real plus Attack")
    if len(lcc_df) != int(fingerprints.get("lcc_candidate_n", -1)):
        raise RuntimeError("Stage 0 LCC candidate count mismatch")
    if len(valid_paths) != int(fingerprints.get("lcc_valid_face_n", -1)):
        raise RuntimeError("Stage 0 LCC valid-face count mismatch")
    valid_path_sha = ordered_path_fingerprint(valid_paths)
    if valid_path_sha != fingerprints.get("lcc_valid_path_fingerprint_sha256"):
        raise RuntimeError("Stage 0 ordered LCC VALID path fingerprint mismatch")
    if stage0_summary.get("lcc_valid_path_fingerprint_sha256") != valid_path_sha:
        raise RuntimeError("Stage 0 summary ordered LCC VALID path fingerprint mismatch")
    return {
        "stage0_dir": stage0,
        "train_keys": train_keys,
        "val_keys": val_keys,
        "train_labels": train_labels,
        "val_labels": val_labels,
        "train_codes": train_codes,
        "val_codes": val_codes,
        "bbox_cache": bbox_cache,
        "lcc_df": lcc_df,
        "lcc_cache": lcc_records,
        "lcc_valid_paths": valid_paths,
        "fingerprints": fingerprints,
    }
def assert_raw_image_paths(resources, celeba_root, lcc_root):
    celeba_keys = list(map(str, resources["train_keys"])) + list(map(str, resources["val_keys"]))
    lcc_paths = resources["lcc_df"]["path"].astype(str).tolist()
    missing_celeba = [str(Path(celeba_root) / key) for key in celeba_keys if not (Path(celeba_root) / key).is_file()]
    missing_lcc = [str(Path(lcc_root) / path) for path in lcc_paths if not (Path(lcc_root) / path).is_file()]
    if missing_celeba or missing_lcc:
        raise FileNotFoundError(
            "Raw PAD image preflight failed before training: CelebA missing={}, LCC missing={}".format(
                missing_celeba[:5], missing_lcc[:5]
            )
        )
    print("Verified raw image paths before training: CelebA selected={}, LCC candidates={}".format(
        len(celeba_keys), len(lcc_paths)
    ))


def runtime_crop_bgr(image, bbox_xyxy, factor):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    bw, bh = x2 - x1, y2 - y1
    if bw <= 0 or bh <= 0:
        raise ValueError(f"Invalid xyxy bbox: {bbox_xyxy}")
    extent = max(bw, bh) * float(factor)
    side = int(extent)
    cx, cy = x1 + bw / 2.0, y1 + bh / 2.0
    xs, ys = int(cx - extent / 2.0), int(cy - extent / 2.0)
    xa, ya, xb, yb = max(0, xs), max(0, ys), min(w, xs + side), min(h, ys + side)
    if xb <= xa or yb <= ya:
        raise RuntimeError("Empty crop intersection")
    crop = image[ya:yb, xa:xb]
    return cv2.copyMakeBorder(
        crop, max(0, -ys), max(0, ys + side - h),
        max(0, -xs), max(0, xs + side - w), cv2.BORDER_REFLECT_101
    )

def resize_letterbox_rgb(rgb, size=224):
    h, w = rgb.shape[:2]
    ratio = size / max(h, w)
    nh, nw = max(1, int(h * ratio)), max(1, int(w * ratio))
    interp = cv2.INTER_LANCZOS4 if ratio > 1 else cv2.INTER_AREA
    image = cv2.resize(rgb, (nw, nh), interpolation=interp)
    dh, dw = size - nh, size - nw
    return cv2.copyMakeBorder(image, dh // 2, dh - dh // 2, dw // 2, dw - dw // 2, cv2.BORDER_REFLECT_101)

def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb

def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]

def rgb_tensor(rgb):
    unit = rgb.astype(np.float32) / 255.0
    unit = (unit - CELEBA_MEAN) / CELEBA_STD
    return torch.from_numpy(np.ascontiguousarray(unit.transpose(2, 0, 1), dtype=np.float32))

def optional_crop_index(stage0_dir):
    path = Path(stage0_dir) / "crop_cache" / "index.json"
    return json.loads(path.read_text()) if path.is_file() else {}

def read_rgb_crop(image_path, record, domain, key, crop_index, stage0_dir, augment=False):
    crop_factor = validate_bbox_record(record, domain, key)
    cache_key = "{}::{}".format(domain, key)
    if not augment and cache_key in crop_index:
        path = Path(stage0_dir) / "crop_cache" / crop_index[cache_key]
        bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if bgr is None:
            raise FileNotFoundError("Optional Stage 0 crop cache entry is unreadable: {}".format(path))
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(image_path)
    box = bbox_jitter(record) if augment else record["bbox_xyxy"]
    crop = runtime_crop_bgr(bgr, box, crop_factor)
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    if augment:
        rgb = apply_e1_mild_augmentation(rgb)
    return resize_letterbox_rgb(rgb, INPUT_SIZE)
class CelebAMicroDataset(Dataset):
    def __init__(self, keys, labels, codes, root, bbox_records, stage0_dir, train=False):
        self.keys = list(map(str, keys))
        self.labels = np.asarray(labels, dtype=np.int64)
        self.codes = np.asarray(codes, dtype=np.int64)
        self.root = Path(root)
        self.records = bbox_records
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.train = bool(train)
        if len(self.keys) != len(self.labels) or len(self.keys) != len(self.codes):
            raise ValueError("CelebA key/label/code lengths differ")
        missing = [k for k in self.keys if k not in self.records]
        if missing:
            raise RuntimeError(f"{len(missing)} CelebA keys lack Stage 0 bbox records")
    def __len__(self):
        return len(self.keys)
    def __getitem__(self, i):
        key = self.keys[i]
        rgb = read_rgb_crop(
            self.root / key, self.records[key], "celeba", key,
            self.crop_index, self.stage0_dir, augment=self.train
        )
        return rgb_tensor(rgb), int(self.labels[i]), key, int(self.codes[i])

class LCCValidDataset(Dataset):
    def __init__(self, frame, root, records, stage0_dir):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.records = records
        self.stage0_dir = Path(stage0_dir)
        self.crop_index = optional_crop_index(stage0_dir)
        self.rows = []
        by_path = {}
        for rec in records.values():
            by_path.setdefault(str(rec.get("path", "")), []).append(rec)
        for _, row in self.frame.iterrows():
            if str(row["status"]) != "VALID":
                continue
            path = str(row["path"])
            candidates = by_path.get(path, [])
            if len(candidates) != 1:
                raise RuntimeError(f"LCC cache join for {path} has {len(candidates)} records")
            self.rows.append((path, int(row["label"]), candidates[0]))
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        key, label, record = self.rows[i]
        rgb = read_rgb_crop(
            self.root / key, record, "lcc", key, self.crop_index,
            self.stage0_dir, augment=False
        )
        return rgb_tensor(rgb), int(label), key, record.get("status", "VALID")

class SpatialE1(nn.Module):
    def __init__(self, imagenet_state=None):
        super().__init__()
        base = models.mobilenet_v3_small(weights=None)
        if imagenet_state is not None:
            missing, unexpected = base.load_state_dict(imagenet_state, strict=False)
            allowed_missing = [k for k in missing if k.startswith("classifier.")]
            bad_missing = [k for k in missing if k not in allowed_missing]
            allowed_unexpected = [k for k in unexpected if k.startswith("classifier.")]
            bad_unexpected = [k for k in unexpected if k not in allowed_unexpected]
            if bad_missing or bad_unexpected:
                raise RuntimeError(
                    "ImageNet MobileNetV3-Small state_dict is incomplete/incompatible; "
                    "backbone missing={}, unexpected={}".format(bad_missing[:5], bad_unexpected[:5])
                )
        self.features = base.features
        self.avgpool = base.avgpool
        self.projection = nn.Sequential(
            nn.Linear(576, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(128, 3),
        )
    def forward_features(self, x):
        x = self.features(x)
        x = self.avgpool(x).flatten(1)
        return self.projection(x)
    def forward(self, x):
        feature = self.forward_features(x)
        return self.classifier(feature)

def load_imagenet_weights(local_path=""):
    if local_path:
        state = torch.load(local_path, map_location="cpu", weights_only=False)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        if not isinstance(state, dict):
            raise RuntimeError("Local ImageNet weights must be a state_dict")
        return state
    try:
        return MobileNet_V3_Small_Weights.DEFAULT.get_state_dict(progress=True)
    except Exception as exc:
        raise RuntimeError(
            "ImageNet MobileNetV3-Small weights are unavailable. Enable Kaggle Internet "
            "or set IMAGENET_WEIGHTS_PATH to a compatible torchvision state_dict. "
            "Random initialization is intentionally not used."
        ) from exc

def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)
def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))
def threshold_from_val(y3, scores):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("Threshold calibration needs non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("CelebA labels must be 0=Real, 1=Physical, 2=Digital")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("Threshold calibration requires at least one Real and one Attack sample")
    if not np.isfinite(scores).all():
        raise ValueError("Threshold calibration scores contain NaN or Infinity")
    values = np.unique(scores)
    lower = np.nextafter(values[0], -np.inf)
    upper = np.nextafter(values[-1], np.inf)
    if not np.isfinite(lower):
        lower = values[0]
    if not np.isfinite(upper):
        upper = values[-1]
    midpoints = values[:-1] / 2.0 + values[1:] / 2.0
    candidates = np.concatenate(([lower], midpoints, [upper]))
    best = None
    for threshold in candidates:
        pred_real = scores >= threshold
        real = y_real == 1
        attack = y_real == 0
        bpcer = float(np.mean(~pred_real[real]))
        apcer = float(np.mean(pred_real[attack]))
        acer = 0.5 * (apcer + bpcer)
        key = (acer, abs(float(threshold)), float(threshold))
        if best is None or key < best[0]:
            best = (key, float(threshold))
    if best is None or not np.isfinite(best[1]):
        raise RuntimeError("Threshold search did not produce a finite threshold")
    return best[1]
def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }
def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores

def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result

def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes
def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full
def trainable_parameter_count(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


In [ ]:
seed_everything(SEED)
resources = load_stage0_resources()
stage0_dir = resources["stage0_dir"]
out_dir = Path(OUTPUT_ROOT) / "stage1_shared_micro_e1"
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
celeba_root = find_celeba_root()
lcc_root = find_lcc_root(stage0_dir / "lcc_external_dev_manifest.csv")
assert_raw_image_paths(resources, celeba_root, lcc_root)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Stage 1 training requires a Kaggle GPU accelerator")
use_amp = bool(AMP and device.type == "cuda")
lcc_candidates = len(resources["lcc_df"])
lcc_valid_n = int((resources["lcc_df"]["status"].astype(str) == "VALID").sum())
lcc_coverage = lcc_valid_n / max(1, lcc_candidates)
assert len(resources["train_keys"]) == 5000 and len(resources["val_keys"]) == 2000
assert len(set(resources["train_keys"]) & set(resources["val_keys"])) == 0

print("=== MICRO SEARCH PREFLIGHT ===")
print("candidate: Stage 1 shared spatial E1")
print("Stage 0 resource directory:", stage0_dir)
print("seed:", SEED)
print("train_n / val_n:", len(resources["train_keys"]), "/", len(resources["val_keys"]))
print("train class counts:", dict(Counter(map(int, resources["train_labels"]))))
print("val class counts:", dict(Counter(map(int, resources["val_labels"]))))
print("parent E1 manifest SHA:", resources["fingerprints"]["parent_e1_manifest_sha256"])
print("micro manifest SHA:", resources["fingerprints"]["micro_manifest_sha256"])
print("shared E1 checkpoint SHA: not created yet")
print("LCC manifest SHA:", resources["fingerprints"]["lcc_manifest_sha256"])
print("LCC cache SHA:", resources["fingerprints"]["lcc_cache_sha256"])
print("CelebA root:", celeba_root)
print("LCC root/cache:", lcc_root, stage0_dir / "lcc_scrfd_cache_micro_v1.json")
print("device / AMP:", device, use_amp)
print("trainable params: pending ImageNet model initialization")

train_ds = CelebAMicroDataset(
    resources["train_keys"], resources["train_labels"], resources["train_codes"],
    celeba_root, resources["bbox_cache"], stage0_dir, train=True
)
val_ds = CelebAMicroDataset(
    resources["val_keys"], resources["val_labels"], resources["val_codes"],
    celeba_root, resources["bbox_cache"], stage0_dir, train=False
)
lcc_ds = LCCValidDataset(resources["lcc_df"], lcc_root, resources["lcc_cache"], stage0_dir)
loader_args = {
    "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
    "pin_memory": True, "persistent_workers": NUM_WORKERS > 0,
}
train_loader = DataLoader(train_ds, shuffle=True, drop_last=False, **loader_args)
val_loader = DataLoader(val_ds, shuffle=False, drop_last=False, **loader_args)
lcc_loader = DataLoader(lcc_ds, shuffle=False, drop_last=False, **loader_args)

print("ImageNet weights source:", IMAGENET_WEIGHTS_PATH or "torchvision cached/downloaded pretrained state_dict")
imagenet_state = load_imagenet_weights(IMAGENET_WEIGHTS_PATH)
model = SpatialE1(imagenet_state).to(device)
print("=== MICRO SEARCH PREFLIGHT COMPLETE ===")
print("candidate: Stage 1 shared spatial E1")
print("seed:", SEED)
print("train_n / val_n:", len(resources["train_keys"]), "/", len(resources["val_keys"]))
print("train class counts:", dict(Counter(map(int, resources["train_labels"]))))
print("val class counts:", dict(Counter(map(int, resources["val_labels"]))))
print("parent E1 manifest SHA:", resources["fingerprints"]["parent_e1_manifest_sha256"])
print("micro manifest SHA:", resources["fingerprints"]["micro_manifest_sha256"])
print("shared E1 checkpoint SHA: not created yet")
print("LCC manifest SHA:", resources["fingerprints"]["lcc_manifest_sha256"])
print("LCC cache SHA:", resources["fingerprints"]["lcc_cache_sha256"])
print("CelebA root:", celeba_root)
print("LCC root/cache:", lcc_root, stage0_dir / "lcc_scrfd_cache_micro_v1.json")
print("device / AMP:", device, use_amp)
print("trainable params:", trainable_parameter_count(model))
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    [
        {"params": list(model.features.parameters()) + list(model.avgpool.parameters()), "lr": 1e-5},
        {"params": list(model.projection.parameters()) + list(model.classifier.parameters()), "lr": 1e-4},
    ],
    weight_decay=1e-4,
)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
history = []
best_acer = float("inf")
best_epoch = None
best_state = None
patience = 2
max_epochs = 6
for epoch in range(1, max_epochs + 1):
    model.train()
    losses = []
    started = time.time()
    for x, y, _keys, _codes in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))
    val_logits, val_y, val_keys, val_codes = eval_spatial(model, val_loader, device, use_amp)
    val_scores = binary_score(torch.as_tensor(val_logits)).cpu().numpy()
    threshold = threshold_from_val(val_y, val_scores)
    val_metrics, _ = metrics_celeba(val_y, val_logits, threshold)
    row = {
        "epoch": epoch, "train_loss": float(np.mean(losses)),
        "val_acer": val_metrics["acer"], "val_auc": val_metrics["auc"],
        "val_apcer": val_metrics["apcer"], "val_bpcer": val_metrics["bpcer"],
        "val_threshold": float(threshold), "seconds": time.time() - started,
    }
    history.append(row)
    print(row)
    if val_metrics["acer"] < best_acer:
        best_acer = val_metrics["acer"]
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    elif epoch >= 3 and epoch - best_epoch >= patience:
        print("Early stop: micro-Val ACER did not improve for", patience, "epochs")
        break

if best_state is None:
    raise RuntimeError("No Stage 1 checkpoint selected")
last_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
model.to(device)
val_logits, val_y, val_keys, val_codes = eval_spatial(model, val_loader, device, use_amp)
assert_ordered_keys(val_keys, resources["val_keys"], "Stage 1 micro-Val")
val_scores = binary_score(torch.as_tensor(val_logits)).cpu().numpy()
locked_threshold = threshold_from_val(val_y, val_scores)
val_metrics, _ = metrics_celeba(val_y, val_logits, locked_threshold)
val_predictions = make_prediction_frame(val_keys, val_y, val_codes, val_logits, locked_threshold)

lcc_logits, lcc_y, lcc_keys, lcc_predictions = eval_lcc_spatial(
    model, lcc_loader, device, use_amp, resources["lcc_df"], locked_threshold
)
lcc_scores = binary_score(torch.as_tensor(lcc_logits)).cpu().numpy() if len(lcc_logits) else np.empty(0)
lcc_metrics = metrics_binary(
    lcc_y, lcc_scores, locked_threshold,
    coverage=lcc_coverage, candidate_n=lcc_candidates
)
lcc_predictions["locked_threshold"] = locked_threshold

out_dir = Path(OUTPUT_ROOT) / "stage1_shared_micro_e1"
out_dir.mkdir(parents=True, exist_ok=True)
best_path = out_dir / "micro_e1_v1_best.pth"
last_path = out_dir / "micro_e1_v1_checkpoint_last.pth"
torch.save({"state_dict": best_state, "epoch": best_epoch, "val_acer": best_acer}, best_path)
torch.save({"state_dict": last_state, "epoch": epoch}, last_path)
checkpoint_sha = sha256_file(best_path)
pd.DataFrame(history).to_csv(out_dir / "micro_e1_v1_training_history.csv", index=False)
val_predictions.to_csv(out_dir / "micro_e1_v1_val_predictions.csv", index=False)
lcc_predictions.to_csv(out_dir / "micro_e1_v1_lcc_predictions.csv", index=False)
config = {
    "schema": "pad_micro_stage1_config_v1", "seed": SEED,
    "architecture": E1_ARCHITECTURE, "input_size": INPUT_SIZE,
    "normalization_mean": CELEBA_MEAN.tolist(), "normalization_std": CELEBA_STD.tolist(),
    "gamma": False,
    "augmentation_implementation": (
        "self-contained RGB flip and numpy brightness/contrast; same probabilities/ranges "
        "and same implementation in Stage 1 and M0-M4; Val/LCC are unaugmented"
    ),
    "augmentation": {
        "horizontal_flip_p": 0.5, "brightness_contrast_p": 0.30,
        "brightness_limit": 0.10, "contrast_limit": 0.10,
        "bbox_jitter_p": 0.20, "bbox_jitter_scale": [0.95, 1.05],
        "bbox_jitter_translate": 0.05,
    },
    "budget": {
        "max_epochs": max_epochs, "patience": patience, "earliest_early_stop_epoch": 3,
        "batch_size": BATCH_SIZE, "amp": use_amp, "weight_decay": 1e-4,
        "backbone_lr": 1e-5, "head_lr": 1e-4, "label_smoothing": 0.1, "grad_clip": 5.0,
    },
    "resource_hashes": resources["fingerprints"],
    "checkpoint_sha256": checkpoint_sha,
    "selected_epoch": int(best_epoch),
    "threshold_source": "CelebA frozen micro-Val only",
}
(out_dir / "micro_e1_v1_config.json").write_text(json.dumps(config, indent=2, allow_nan=False) + "\n")
meta = {
    "schema": "pad_micro_stage1_meta_v1",
    "architecture": E1_ARCHITECTURE,
    "checkpoint_file": best_path.name,
    "checkpoint_sha256": checkpoint_sha,
    "selected_epoch": int(best_epoch),
    "locked_validation_logit_threshold": float(locked_threshold),
    "threshold_source": "CelebA frozen micro-Val only",
    "resource_hashes": resources["fingerprints"],
    "parameter_count": int(sum(p.numel() for p in model.parameters())),
}
(out_dir / "micro_e1_v1_meta.json").write_text(json.dumps(meta, indent=2, allow_nan=False) + "\n")
summary = {
    "schema": "pad_micro_stage1_summary_v1",
    "lcc_dataset_role": "LCC-FASD external-dev / cross-domain stress set",
    "resource_hashes": resources["fingerprints"],
    "architecture": E1_ARCHITECTURE,
    "checkpoint_sha256": checkpoint_sha,
    "locked_validation_logit_threshold": float(locked_threshold),
    "val": val_metrics, "lcc": lcc_metrics,
    "parameter_count": meta["parameter_count"],
    "lcc_candidate_n": lcc_candidates, "lcc_valid_face_n": lcc_valid_n,
}
(out_dir / "micro_e1_v1_summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n")
stage1_zip = Path(OUTPUT_ROOT) / "pad_micro_stage1_shared_e1.zip"
write_zip(out_dir, stage1_zip)
print("\nShared Stage 1 checkpoint SHA256:", checkpoint_sha)
print("Micro-Val locked threshold:", locked_threshold)
print("Micro-Val metrics:", val_metrics)
print("LCC external-dev metrics at the locked threshold:", lcc_metrics)
print("Stage 1 ZIP:", stage1_zip)
